In [15]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.preprocessing import StandardScaler

def load_and_clean_data(data_directory):
    
    # print("Generating column names for features...")
    column_names = ['timestamp']
    for i in range(100):
        column_names.extend([f'{i}x', f'{i}y', f'{i}z'])
        
    all_features = []
    all_labels = []
    
    file_pattern = os.path.join(data_directory, '*_datapoints.txt') 
    datapoint_files = glob.glob(file_pattern) 
    print(f"Found {len(datapoint_files)} datapoint files in {data_directory}")
    for feature_file in datapoint_files:
        df_features = pd.read_csv(feature_file, sep=r'\s+', skiprows=1, header=None, names=column_names)
        target_file = feature_file.replace('datapoints', 'target') 
        if not os.path.exists(target_file):
            target_file = feature_file.replace('datapoints', 'targets')

        df_targets = pd.read_csv(target_file, header=None, names=['is_active'])
    

        expression_name = os.path.basename(feature_file).split('_')[1] 
        labels = np.where(df_targets['is_active'] > 0, expression_name, 'neutral') 
        
        all_features.append(df_features)
        all_labels.extend(labels)

    X = pd.concat(all_features, ignore_index=True) 
    y = np.array(all_labels)

    X = X.drop(columns=['timestamp'])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X) 

    
    return X_scaled, y




In [16]:
print("Loading, cleaning, and mapping dataset based on target files...")
X, y = load_and_clean_data(data_directory='datasets')
print(f"Total frames (rows): {X.shape[0]}")
print(f"Total features (columns): {X.shape[1]}")

unique, counts = np.unique(y, return_counts=True)
print("\nClass Distribution:")
for label, count in zip(unique, counts): 
    print(f"{label}: {count} frames")

print("\nFirst 10 rows of X:")
print(X[:10])

print("\nFirst 10 labels:")
print(y[:100])

print(X.shape)


Loading, cleaning, and mapping dataset based on target files...
Found 18 datapoint files in datasets
Total frames (rows): 27936
Total features (columns): 300

Class Distribution:
affirmative: 942 frames
conditional: 1137 frames
doubt: 1271 frames
emphasis: 861 frames
negative: 1240 frames
neutral: 18059 frames
relative: 1194 frames
topics: 827 frames
wh: 1158 frames
yn: 1247 frames

First 10 rows of X:
[[-4.22103327 -0.64668094 -0.71377239 ... -3.26701857 -1.13192617
  -0.22807189]
 [-2.74119749 -1.47702869 -0.34239866 ... -2.08759967 -1.76525053
  -2.60250991]
 [-2.26820079 -1.6734285  -0.20735367 ... -1.71589864 -1.93266872
   0.14475449]
 ...
 [-1.46962769 -1.71690989  0.20698892 ... -1.2452104  -1.83417384
  -2.60250991]
 [-1.47233967 -1.73931747  0.22233494 ... -1.27707261 -1.8619742
   0.44767592]
 [-1.49181028 -1.77719696  0.22233494 ... -1.30294678 -1.85488934
   0.45932675]]

First 10 labels:
['neutral' 'neutral' 'neutral' 'neutral' 'neutral' 'neutral' 'neutral'
 'neutral' 'ne

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import time

def run_classification(X, y):
    print("\n--- Starting Classification ---")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training on {len(X_train)} frames, Testing on {len(X_test)} frames...\n")

    models = {
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Support Vector Machine (SVM)": SVC(kernel='rbf', random_state=42)
    }
    
    for name, clf in models.items():
        print(f"Evaluating {name}")
        start_time = time.time()
       
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        elapsed_time = time.time() - start_time
        accuracy = accuracy_score(y_test, y_pred)
        print(f"Time to Train & Predict: {elapsed_time:.2f} seconds")
        print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
        
        print(classification_report(y_test, y_pred))
        print("\n")


run_classification(X, y)


--- Starting Classification ---
Training on 22348 frames, Testing on 5588 frames...

Evaluating Random Forest
Time to Train & Predict: 9.44 seconds
Model Accuracy: 93.13%

              precision    recall  f1-score   support

 affirmative       0.86      0.78      0.82       188
 conditional       0.93      0.82      0.87       228
       doubt       0.92      0.91      0.92       254
    emphasis       0.91      0.87      0.89       172
    negative       0.93      0.83      0.88       248
     neutral       0.94      0.97      0.95      3612
    relative       0.91      0.88      0.90       239
      topics       0.92      0.80      0.86       165
          wh       0.95      0.91      0.93       232
          yn       0.93      0.90      0.92       250

    accuracy                           0.93      5588
   macro avg       0.92      0.87      0.89      5588
weighted avg       0.93      0.93      0.93      5588



Evaluating Support Vector Machine (SVM)
Time to Train & Predict: 8

In [18]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def run_clustering(X, y):
    print("\nStarting Clustering")

    print("Compressing 300 features into 2 dimensions using PCA...")
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    print("Running K-Means Clustering to find 10 groups...")
    kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)

    cluster_labels = kmeans.fit_predict(X_pca)
   
    sil_score = silhouette_score(X_pca, cluster_labels)
    print(f"\nSilhouette Score: {sil_score:.4f}")


    unique, counts = np.unique(y, return_counts=True)
    print("\nClass Distribution:")
    for label, count in zip(unique, counts): 
        print(f"{label}: {count} frames")
    
    print("\nCluster Distribution:")
    unique, counts = np.unique(cluster_labels, return_counts=True)
    for label, count in zip(unique, counts):
        print(f"{label}: {count} frames")
    

run_clustering(X, y)


Starting Clustering
Compressing 300 features into 2 dimensions using PCA...
Running K-Means Clustering to find 10 groups...

Silhouette Score: 0.3744

Class Distribution:
affirmative: 942 frames
conditional: 1137 frames
doubt: 1271 frames
emphasis: 861 frames
negative: 1240 frames
neutral: 18059 frames
relative: 1194 frames
topics: 827 frames
wh: 1158 frames
yn: 1247 frames

Cluster Distribution:
0: 5184 frames
1: 1959 frames
2: 1129 frames
3: 3340 frames
4: 1048 frames
5: 2657 frames
6: 1145 frames
7: 2330 frames
8: 4336 frames
9: 4808 frames
